# Notebook to crossmatch an external catalog with the ALeRCE database

```Author: Francisco Förster, modifications: Kay Medina, Alejandra Muñoz Arancibia. Last updated: 20260626```

*It is highly recommended that you try this notebook in Google Colab using the following [link](https://colab.research.google.com/github/alercebroker/usecases/blob/master/notebooks/ZTF/ALeRCE_ExternalCatalog_Xmatch.ipynb).*
This will avoid you from having to sort out library installation problems and focus on the contents of the tutorial. You can try installing the dependencies later in your own system.

In [1]:
import pandas as pd

In [2]:
#!pip install pyvo
import pyvo

### TAP connection setup

Instead of using a direct PostgreSQL connection, we connect to the ALeRCE ZTF legacy database using Astronomical Data Query Language ([ADQL](https://www.ivoa.net/documents/ADQL/)) queries to the ALeRCE TAP service, returning results as pandas dataframes. Database table names are prefixed with `ztf.` (e.g. `ztf.object`, `ztf.detection`, etc.).

In [3]:
TAP_URL = "https://tap.alerce.online/tap"
tap_service = pyvo.dal.TAPService(TAP_URL)

def tap_query(query, maxrec=10_000_000):
    result = tap_service.search(query, maxrec=maxrec)
    return result.to_table().to_pandas()

## Example external catalog

In [4]:
# The catalog should have id_source, ra, and dec columns
df = pd.read_csv("https://github.com/alercebroker/usecases/blob/master/example_data/watchlist.csv?raw=True")
df.head()

,id_source,ra,dec
0,source_1,254.843418,1.646404
1,source_2,215.416280,13.229551


### Crossmatch via ADQL cone search (per position)

A previous version of this notebook used a PostgreSQL-specific `q3c_join` plus a `WITH ... (VALUES ...)` common table expression (CTE) to crossmatch the input catalog against the database in a single query. Neither is part of ADQL, so on TAP we run one ADQL cone search per input position (`CONTAINS`/`CIRCLE`), letting the service do the spatial match server-side on its q3c index, and concatenate the results.

Note: batching all positions into one query does not work on this service: OR-ing many `CONTAINS` defeats the q3c index (504 timeout), and a `TAP_UPLOAD` spatial join against the foreign `ztf.object` table cannot push the join down to the remote DB (it hangs). The per-position cone search below is the approach that actually uses the spatial index.

In [5]:
def ztf_crossmatch(df, search_radius=1):
    '''
    df: external catalog dataframe (with columns id_source, ra, dec)
    search_radius: crossmatch radius in arcsec (default=1)

    The output is a dataframe with the source id, ra, and dec, as well as the
    ALeRCE database oid, meanra, meandec, the crossmatch distance in arcsec
    (dist_arcsec) and the time of first detection (firstmjd).

    TAP/ADQL has no q3c_join and no inline-VALUES CTE, so we run one ADQL cone
    search per input position (CONTAINS/CIRCLE, matched server-side on the q3c
    index) and concatenate the results.
    '''
    # CIRCLE wants the radius in degrees; the catalog convention here is arcsec.
    radius_deg = search_radius / 3600

    frames = []
    for _, row in df.iterrows():
        query = f"""
        SELECT
            '{row.id_source}' AS source_id, {row.ra} AS ra, {row.dec} AS dec,
            o.oid, o.meanra, o.meandec, o.firstmjd,
            3600 * DISTANCE(POINT('ICRS', {row.ra}, {row.dec}),
                            POINT('ICRS', o.meanra, o.meandec)) AS dist_arcsec
        FROM ztf.object AS o
        WHERE CONTAINS(POINT('ICRS', o.meanra, o.meandec),
                       CIRCLE('ICRS', {row.ra}, {row.dec}, {radius_deg})) = 1
        """
        frames.append(tap_query(query))

    return pd.concat(frames, ignore_index=True)

## Do the crossmatch

Each position is a separate cone search (~a few seconds each on the q3c index), so the total time
grows with the size of your catalog. For a large catalog, expect roughly that per-position cost
times the number of rows.

In [6]:
results = ztf_crossmatch(df)
results

,source_id,ra,dec,oid,meanra,meandec,firstmjd,dist_arcsec
0,source_1,254.843418,1.646404,ZTF26aauxhbe,254.843416,1.646405,61167.339722,0.008017
1,source_2,215.416280,13.229551,ZTF26aasmsqg,215.416280,13.229551,61145.361192,0.001487
